In [6]:
from dotenv import load_dotenv
load_dotenv()
import os

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [ ]:
## data ingestion -- from the website, scrape the data

from langchain_community.document_loaders import WebBaseLoader




In [9]:
loader = WebBaseLoader("https://docs.langchain.com/oss/python/langchain/human-in-the-loop")

loader


In [11]:
docs = loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/langchain/human-in-the-loop', 'title': 'Human-in-the-loop - Docs by LangChain', 'language': 'en'}, page_content='\n\n\n\n\n\nHuman-in-the-loop - Docs by LangChain\n\n\n\n\n\n\n\n\n\nDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentWould you notice if your agent got worse in production? Tell us in our annual survey for the chance to win LangChain swag.Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationAdvanced usageHuman-in-the-loopOverviewManaged Deep AgentsDeep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessagesToolsShort-term memoryEvent streamingStreamingStructured outputMiddlewareOverviewPrebuilt middlewareCustom middlewareFrontendOverviewPatte

In [12]:
## load data -> docs -> divide into chunks --> convert into vector using embeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
docs = text_splitter.split_documents(docs)
docs

[Document(metadata={'source': 'https://docs.langchain.com/oss/python/langchain/human-in-the-loop', 'title': 'Human-in-the-loop - Docs by LangChain', 'language': 'en'}, page_content='Human-in-the-loop - Docs by LangChain'),
 Document(metadata={'source': 'https://docs.langchain.com/oss/python/langchain/human-in-the-loop', 'title': 'Human-in-the-loop - Docs by LangChain', 'language': 'en'}, page_content='Documentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentWould you notice if your agent got worse in production? Tell us in our annual survey for the chance to win LangChain swag.Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationAdvanced usageHuman-in-the-loopOverviewManaged Deep AgentsDeep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgen

In [15]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434"
)

embeddings

OllamaEmbeddings(model='nomic-embed-text', dimensions=None, validate_model_on_init=False, base_url='http://localhost:11434', client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

In [16]:
from langchain_community.vectorstores import FAISS

vectorstore_db = FAISS.from_documents(docs, embeddings)
vectorstore_db

In [ ]:
query='What is Human-in-the-Loop?'
response = vectorstore_db.similarity_search(query)
response

[Document(id='c002d94e-6dc2-4914-ae70-530eea4ea65b', metadata={'source': 'https://docs.langchain.com/oss/python/langchain/human-in-the-loop', 'title': 'Human-in-the-loop - Docs by LangChain', 'language': 'en'}, page_content='Human-in-the-loop - Docs by LangChain'),
 Document(id='fe5bf9af-8ace-467a-b72b-9d8918f88360', metadata={'source': 'https://docs.langchain.com/oss/python/langchain/human-in-the-loop', 'title': 'Human-in-the-loop - Docs by LangChain', 'language': 'en'}, page_content='# Human-in-the-loop leverages LangGraph\'s persistence layer.\n# You must provide a thread ID to associate the execution with a conversation thread,\n# so the conversation can be paused and resumed (as is needed for human review).\nconfig = {"configurable": {"thread_id": "some_id"}}\n# Run the graph until the interrupt is hit.\nresult = agent.invoke(\n    {\n        "messages": [\n            {\n                "role": "user",\n                "content": "Delete old records from the database",\n         

In [20]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

print(llm)

metadata={'lc_versions': {'langchain-core': '1.8.3', 'langchain': '1.5.0'}} model='llama3.2' temperature=0.0


In [23]:
## Retrieval Chain -> When we provide context with the question.
## Document Chain

from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompts = ChatPromptTemplate.from_template(
    """
    Answer the following question based only on the provided context::
    <context>
    {context}
    </context>
    """
)

document_chain = create_stuff_documents_chain(llm, prompts)
document_chain


RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on the provided context::\n    <context>\n    {context}\n    </context>\n    '), additional_kwargs={})])
| ChatOllama(metadata={'lc_versions': {'langchain-core': '1.8.3', 'langchain': '1.5.0'}}, model='llama3.2', temperature=0.0)
| StrOutputParser(), kwargs={}, config={'run_name': 'stuff_documents_chain'}, config_factories=[])

In [25]:
from langchain_core.documents import Document

document_chain.invoke({
    "input" : "Each interrupt also carries a response_schema",
    "context" : [Document(page_content="a JSON schema with one branch per decision the tool allows, matched on the decision’s type. ")]
})

"The tool allows a JSON schema with one branch per decision, where each branch is matched on the decision's type."

In [26]:
retriever = vectorstore_db.as_retriever()

from langchain_classic.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(retriever, document_chain)
retrieval_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000289B0362A50>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on the provided context::\n    <context>\n    {context}\n    </context>\n    '), additional_kwargs={})

In [29]:
## Get the response
response = retrieval_chain.invoke({
    "input" : "The Human-in-the-Loop (HITL) middleware lets you add "
})
response["answer"]

'The middleware is configured to pause execution once per tool call, which means that the agent will wait for human input after each tool call.'